In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import tensorflow as tf

SQUEEZE_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/models/"
    "squeezenet_baseline.keras"
)

VAL_CSV = (
    "/content/drive/MyDrive/ASD_XAI_Project/metadata/"
    "val_slices.csv"
)

squeezenet_model = tf.keras.models.load_model(
    SQUEEZE_PATH
)

val_slices_df = pd.read_csv(VAL_CSV)

print("Model:", squeezenet_model.input_shape)
print("Validation slices:", len(val_slices_df))
print("Subjects:", val_slices_df["SUB_ID"].nunique())

Mounted at /content/drive
Model: (None, 224, 224, 3)
Validation slices: 1054
Subjects: 6


In [ ]:
IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )
    image = tf.cast(
        image,
        tf.float32
    )

    return image

In [ ]:
last_conv_layer = None

for layer in reversed(squeezenet_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer
        break

print("Last convolutional layer:")
print(last_conv_layer.name)
print(last_conv_layer.output.shape)

Last convolutional layer:
conv2d_25
(None, 14, 14, 1)


In [ ]:
def make_gradcam_heatmap(image, model, conv_layer_name):

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(conv_layer_name).output,
            model.output
        ]
    )

    image_batch = tf.expand_dims(image, axis=0)

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(
            image_batch,
            training=False
        )

        score = predictions[:, 0]

    grads = tape.gradient(
        score,
        conv_outputs
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    heatmap = tf.maximum(
        heatmap,
        0
    )

    max_value = tf.reduce_max(heatmap)

    if max_value > 0:
        heatmap = heatmap / max_value

    return heatmap.numpy()


test_image = load_image(
    val_slices_df.iloc[0]["SLICE_PATH"]
)

test_heatmap = make_gradcam_heatmap(
    test_image,
    squeezenet_model,
    last_conv_layer.name
)

print("Image shape:", test_image.shape)
print("Grad-CAM shape:", test_heatmap.shape)
print(
    "Grad-CAM range:",
    test_heatmap.min(),
    test_heatmap.max()
)

print("Grad-CAM test successful!")

Image shape: (224, 224, 3)
Grad-CAM shape: (14, 14)
Grad-CAM range: 0.9880349 1.0
Grad-CAM test successful!


In [ ]:
def get_gradcam_score(image, model, conv_layer_name):

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(conv_layer_name).output,
            model.output
        ]
    )

    image_batch = tf.expand_dims(image, axis=0)

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(
            image_batch,
            training=False
        )

        score = predictions[:, 0]

    grads = tape.gradient(
        score,
        conv_outputs
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    conv_outputs = conv_outputs[0]

    raw_heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    # Keep only positive influence
    raw_heatmap = tf.maximum(
        raw_heatmap,
        0
    )

    # One Grad-CAM importance score for this MRI slice
    gradcam_score = tf.reduce_mean(
        raw_heatmap
    )

    return float(gradcam_score.numpy())

In [ ]:
for i in range(5):

    image = load_image(
        val_slices_df.iloc[i]["SLICE_PATH"]
    )

    score = get_gradcam_score(
        image,
        squeezenet_model,
        last_conv_layer.name
    )

    print(
        f"Slice {i} Grad-CAM score:",
        score
    )

Slice 0 Grad-CAM score: 0.0026676415000110865
Slice 1 Grad-CAM score: 0.0026676447596400976
Slice 2 Grad-CAM score: 0.002667659427970648
Slice 3 Grad-CAM score: 0.0026676945853978395
Slice 4 Grad-CAM score: 0.0026677323039621115


In [ ]:
conv_layers = []

for layer in squeezenet_model.layers:
    if isinstance(layer, tf.keras.layers.Conv2D):
        conv_layers.append(
            (layer.name, layer.output.shape)
        )

print("Last convolution layers:")

for name, shape in conv_layers[-10:]:
    print(name, shape)

Last convolution layers:
conv2d_16 (None, 14, 14, 48)
conv2d_17 (None, 14, 14, 192)
conv2d_18 (None, 14, 14, 192)
conv2d_19 (None, 14, 14, 64)
conv2d_20 (None, 14, 14, 256)
conv2d_21 (None, 14, 14, 256)
conv2d_22 (None, 14, 14, 64)
conv2d_23 (None, 14, 14, 256)
conv2d_24 (None, 14, 14, 256)
conv2d_25 (None, 14, 14, 1)


In [ ]:
GRADCAM_LAYER = "conv2d_24"

for i in range(5):

    image = load_image(
        val_slices_df.iloc[i]["SLICE_PATH"]
    )

    score = get_gradcam_score(
        image,
        squeezenet_model,
        GRADCAM_LAYER
    )

    print(
        f"Slice {i} Grad-CAM score:",
        score
    )

Slice 0 Grad-CAM score: 7.017069583525881e-05
Slice 1 Grad-CAM score: 7.017342431936413e-05
Slice 2 Grad-CAM score: 7.01850512996316e-05
Slice 3 Grad-CAM score: 7.021324563538656e-05
Slice 4 Grad-CAM score: 7.024336082395166e-05


In [ ]:
def integrated_gradients(
    image,
    model,
    baseline=None,
    steps=32
):

    image = tf.cast(image, tf.float32)

    if baseline is None:
        baseline = tf.zeros_like(image)

    alphas = tf.linspace(
        0.0,
        1.0,
        steps + 1
    )

    interpolated_images = (
        baseline[None, ...]
        + alphas[:, None, None, None]
        * (image[None, ...] - baseline[None, ...])
    )

    with tf.GradientTape() as tape:

        tape.watch(interpolated_images)

        predictions = model(
            interpolated_images,
            training=False
        )[:, 0]

    gradients = tape.gradient(
        predictions,
        interpolated_images
    )

    avg_gradients = tf.reduce_mean(
        gradients[:-1],
        axis=0
    )

    integrated_grads = (
        image - baseline
    ) * avg_gradients

    return integrated_grads


def get_ig_score(image, model):

    attributions = integrated_gradients(
        image,
        model,
        steps=32
    )

    score = tf.reduce_mean(
        tf.abs(attributions)
    )

    return float(score.numpy())


for i in range(5):

    image = load_image(
        val_slices_df.iloc[i]["SLICE_PATH"]
    )

    score = get_ig_score(
        image,
        squeezenet_model
    )

    print(
        f"Slice {i} IG score:",
        score
    )

Slice 0 IG score: 3.5534124254166244e-11
Slice 1 IG score: 1.200994442784875e-10
Slice 2 IG score: 1.807951149901399e-10
Slice 3 IG score: 3.0316676946640086e-10
Slice 4 IG score: 4.274723353070442e-10


In [ ]:
def integrated_gradients(
    image,
    model,
    baseline=None,
    steps=32
):

    image = tf.cast(image, tf.float32)

    if baseline is None:
        baseline = tf.zeros_like(image)

    alphas = tf.linspace(
        0.0,
        1.0,
        steps + 1
    )

    interpolated_images = (
        baseline[None, ...]
        + alphas[:, None, None, None]
        * (image[None, ...] - baseline[None, ...])
    )

    with tf.GradientTape() as tape:

        tape.watch(interpolated_images)

        predictions = model(
            interpolated_images,
            training=False
        )[:, 0]

    gradients = tape.gradient(
        predictions,
        interpolated_images
    )

    avg_gradients = tf.reduce_mean(
        gradients[:-1],
        axis=0
    )

    integrated_grads = (
        image - baseline
    ) * avg_gradients

    return integrated_grads


def get_ig_score(image, model):

    attributions = integrated_gradients(
        image,
        model,
        steps=32
    )

    score = tf.reduce_mean(
        tf.abs(attributions)
    )

    return float(score.numpy())


for i in range(5):

    image = load_image(
        val_slices_df.iloc[i]["SLICE_PATH"]
    )

    score = get_ig_score(
        image,
        squeezenet_model
    )

    print(
        f"Slice {i} IG score:",
        score
    )

Slice 0 IG score: 3.5534124254166244e-11
Slice 1 IG score: 1.200994442784875e-10
Slice 2 IG score: 1.807951149901399e-10
Slice 3 IG score: 3.0316676946640086e-10
Slice 4 IG score: 4.274723353070442e-10


In [ ]:
IG_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_06_squeezenet_gradcam_ig/"
    "squeezenet_ig_validation_scores.csv"
)

os.makedirs(
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_06_squeezenet_gradcam_ig",
    exist_ok=True
)

ig_results = []

for idx, row in val_slices_df.iterrows():

    image = load_image(
        row["SLICE_PATH"]
    )

    score = get_ig_score(
        image,
        squeezenet_model
    )

    ig_results.append({
        "SUB_ID": row["SUB_ID"],
        "SLICE_PATH": row["SLICE_PATH"],
        "LABEL": row["LABEL"],
        "IG_SCORE": score
    })

    if (idx + 1) % 50 == 0:
        print(
            f"IG processed: "
            f"{idx + 1}/{len(val_slices_df)}"
        )


ig_scores_df = pd.DataFrame(
    ig_results
)

ig_scores_df.to_csv(
    IG_OUTPUT,
    index=False
)

print("\nIntegrated Gradients completed!")
print("Rows:", len(ig_scores_df))

print("\nTop IG-ranked slices:")

print(
    ig_scores_df
    .sort_values(
        "IG_SCORE",
        ascending=False
    )
    .head(10)
)

IG processed: 50/1054
IG processed: 100/1054
IG processed: 150/1054
IG processed: 200/1054
IG processed: 250/1054
IG processed: 300/1054
IG processed: 350/1054
IG processed: 400/1054
IG processed: 450/1054
IG processed: 500/1054
IG processed: 550/1054
IG processed: 600/1054
IG processed: 650/1054
IG processed: 700/1054
IG processed: 750/1054
IG processed: 800/1054
IG processed: 850/1054
IG processed: 900/1054
IG processed: 950/1054
IG processed: 1000/1054
IG processed: 1050/1054

Integrated Gradients completed!
Rows: 1054

Top IG-ranked slices:
     SUB_ID                                         SLICE_PATH    LABEL  \
324   51460  /content/drive/MyDrive/ASD_XAI_Project/slices/...      ASD   
323   51460  /content/drive/MyDrive/ASD_XAI_Project/slices/...      ASD   
845   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
321   51460  /content/drive/MyDrive/ASD_XAI_Project/slices/...      ASD   
850   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   

In [12]:
K_VALUES = [5, 10, 15, 20, 30, 40, 50]

ig_topk_results = []

# Rank IG scores separately within each subject
ig_ranked_df = ig_scores_df.copy()

ig_ranked_df["IG_RANK"] = (
    ig_ranked_df
    .groupby("SUB_ID")["IG_SCORE"]
    .rank(
        method="first",
        ascending=False
    )
)

for k in K_VALUES:

    selected = ig_ranked_df[
        ig_ranked_df["IG_RANK"] <= k
    ].copy()

    subject_results = []

    for sub_id, group in selected.groupby("SUB_ID"):

        images = np.stack([
            load_image(path).numpy()
            for path in group["SLICE_PATH"]
        ])

        predictions = squeezenet_model.predict(
            images,
            batch_size=16,
            verbose=0
        ).reshape(-1)

        probability = np.mean(predictions)

        predicted_label = (
            "ASD"
            if probability >= 0.5
            else "Control"
        )

        true_label = group["LABEL"].iloc[0]

        subject_results.append({
            "SUB_ID": sub_id,
            "TRUE_LABEL": true_label,
            "PREDICTED_LABEL": predicted_label,
            "PROBABILITY": probability,
            "NUM_SLICES": len(group)
        })

    subject_df = pd.DataFrame(subject_results)

    accuracy = (
        subject_df["TRUE_LABEL"]
        == subject_df["PREDICTED_LABEL"]
    ).mean()

    ig_topk_results.append({
        "TOP_K": k,
        "TOTAL_SLICES": len(selected),
        "ACCURACY": accuracy
    })

    print(
        f"K={k:2d} | "
        f"Slices={len(selected):3d} | "
        f"Accuracy={accuracy:.4f}"
    )


ig_topk_results_df = pd.DataFrame(
    ig_topk_results
)

print("\n=== IG TOP-K RESULTS ===")
print(ig_topk_results_df)

best_ig = ig_topk_results_df.loc[
    ig_topk_results_df["ACCURACY"].idxmax()
]

print("\n=== BEST IG RESULT ===")
print(best_ig)

K= 5 | Slices= 30 | Accuracy=0.5000
K=10 | Slices= 60 | Accuracy=0.5000
K=15 | Slices= 90 | Accuracy=0.5000
K=20 | Slices=120 | Accuracy=0.5000
K=30 | Slices=180 | Accuracy=0.5000
K=40 | Slices=240 | Accuracy=0.5000
K=50 | Slices=300 | Accuracy=0.5000

=== IG TOP-K RESULTS ===
   TOP_K  TOTAL_SLICES  ACCURACY
0      5            30       0.5
1     10            60       0.5
2     15            90       0.5
3     20           120       0.5
4     30           180       0.5
5     40           240       0.5
6     50           300       0.5

=== BEST IG RESULT ===
TOP_K            5.0
TOTAL_SLICES    30.0
ACCURACY         0.5
Name: 0, dtype: float64


In [13]:
import os

EXP6_DIR = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_06_squeezenet_gradcam_ig"
)

os.makedirs(EXP6_DIR, exist_ok=True)

# Save IG Top-K experiment
ig_topk_results_df.to_csv(
    f"{EXP6_DIR}/ig_topk_results.csv",
    index=False
)

# Save complete IG ranking
ig_ranked_df.to_csv(
    f"{EXP6_DIR}/ig_complete_ranking.csv",
    index=False
)

print("Experiment 06 saved successfully!")
print("Folder:", EXP6_DIR)

print("\nBest IG result:")
print(best_ig)

Experiment 06 saved successfully!
Folder: /content/drive/MyDrive/ASD_XAI_Project/experiment_06_squeezenet_gradcam_ig

Best IG result:
TOP_K            5.0
TOTAL_SLICES    30.0
ACCURACY         0.5
Name: 0, dtype: float64
